# Area Analysis

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import pyrootutils
from xenocanto3 import Quality

ROOT = pyrootutils.setup_root(
    search_from=Path.cwd(),
    indicator="pyproject.toml",
    pythonpath=True,
    dotenv=True,
)

# Parameters

In [ ]:
LAT, LON = 48.8566, 2.3522            # Paris
RADIUS_KM = 50
SPECIES_NAMES = ["Emberiza calandra","Hippolais polyglotta","Regulus ignicapilla"]
MIN_QUALITY = Quality.B
MIN_RECORDINGS = 10                   # drop species below this many recordings from the taxonomy breakdown

# BirdNET pass
MAX_BIRDNET_RECORDINGS = 500          # cap downloads
BIRDNET_MIN_CONFIDENCE = 0.85

# Bounding box

In [ ]:
from building.analysis import bbox_from_radius

bbox = bbox_from_radius(LAT, LON, RADIUS_KM)
print(f"bbox (lat_min, lon_min, lat_max, lon_max): {tuple(round(x, 4) for x in bbox)}")
print(f"lat span: {bbox[2] - bbox[0]:.3f} deg   lon span: {bbox[3] - bbox[1]:.3f} deg")

# Per-species global heatmap

In [ ]:
from building.analysis import fetch_species_global, recordings_to_df, build_heatmap, ANALYSIS_DIR

global_recs = await fetch_species_global(SPECIES_NAMES, min_quality=MIN_QUALITY)
per_species_df = {name: recordings_to_df(recs) for name, recs in global_recs.items()}

for name, df in per_species_df.items():
    print(f"{name:30s} {len(df):5d} recordings (with coords: {df['lat'].notna().sum()})")

heatmap = build_heatmap(per_species_df, center=(LAT, LON), bbox=bbox)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
heatmap.save(str(ANALYSIS_DIR / "heatmap.html"))
heatmap

# Recordings inside the area

In [ ]:
from building.analysis import fetch_area_recordings

area_recs = await fetch_area_recordings(bbox, min_quality=MIN_QUALITY)
area_df = recordings_to_df(area_recs)
print(f"{len(area_df)} recordings, {area_df['scientific_name'].nunique()} unique species")

# Are the target species present in the area?

In [ ]:
from building.analysis import presence_in_area

presence_in_area(SPECIES_NAMES, area_df)

# Taxonomy breakdown of the area

In [ ]:
from building.analysis import taxonomy_breakdown

tax = taxonomy_breakdown(area_df, min_recordings=MIN_RECORDINGS)
totals = tax["totals"].iloc[0]
print(
    f"{totals['n_recordings']} recordings  |  "
    f"{totals['n_species']} species across {totals['n_genera']} genera, "
    f"{totals['n_families']} families, {totals['n_orders']} orders"
    f"  (species with >= {MIN_RECORDINGS} recordings)"
)
if totals["n_unmatched_in_taxonomy"]:
    print(f"({totals['n_unmatched_in_taxonomy']} recordings could not be matched to the eBird taxonomy)")

In [ ]:
tax["by_order"]

In [ ]:
tax["by_family"].head(20)

In [ ]:
tax["by_genus"].head(20)

# Co-occurring species

In [ ]:
from building.analysis import co_occurrence_counts

co_occurrence_counts(area_df, top_n=30)

# Annotations

In [ ]:
from building.analysis import annotation_summary

ann = annotation_summary(area_df)
print(f"{len(ann['annotations'])} annotations across {len(ann['by_species'])} species")
ann["by_species"].head(20)

In [ ]:
ann["annotations"].head(20)

# BirdNET pass on area recordings

In [ ]:
from building.analysis import area_audio_cache_dir, birdnet_area_analysis

bn = await birdnet_area_analysis(
    area_df,
    max_recordings=MAX_BIRDNET_RECORDINGS,
    min_confidence=BIRDNET_MIN_CONFIDENCE,
    cache_dir=area_audio_cache_dir(LAT, LON, RADIUS_KM),
)
det = bn["detections"]
print(
    f"{len(det)} detections across {det['xc_id'].nunique()} recordings  |  "
    f"{bn['by_species'].shape[0]} unique species detected  |  "
    f"{bn['hidden'].shape[0]} species not declared in XC `also`"
)

# All area species, by source

In [ ]:
from building.analysis import combined_species_table

combined = combined_species_table(area_df, bn["detections"], min_recordings=MIN_RECORDINGS)
print(f"{len(combined)} species ( {MIN_RECORDINGS} recordings of any source)")
combined

# Longest recordings in the area

In [ ]:
from building.analysis import longest_recordings

longest_recordings(area_df, center=(LAT, LON), top_n=10)